## V0: Legacy codes for benchmarks in the paper

In [20]:
import os
import pandas as pd
from benchbench.data import cardinal_benchmark_list
from benchbench.data import load_cardinal_benchmark
from benchbench.utils.base import rankdata
from benchbench.measures.cardinal import get_sensitivity

model_col_name = {
    "BigBenchHard": "Model",
    "GLUE": "Model",
    "MMLU": "model_name",
    "MTEB": "Model",
    "OpenLLM": "Model",
    "SuperGLUE": "Model",
    "VTAB": None,
}
dir_out_before = "./data/before"
dir_out_after = "./data/after"
os.makedirs(dir_out_before, exist_ok=True)
os.makedirs(dir_out_after, exist_ok=True)
for bench in cardinal_benchmark_list:
    data, cols = load_cardinal_benchmark(bench, do_rerank=False)
    cols = list(cols)
    data["Average"] = data[cols].mean(axis=1)
    data["Rank"] = rankdata(-data["Average"])
    mc = model_col_name[bench]
    if mc is None:
        data["Model"] = data.index
    else:
        data["Model"] = data[mc]
    data["Model"] = data["Model"].apply(lambda s: s.replace(",", " "))
    data = data.sort_values("Rank")
    data = data.reset_index()
    data[["Model", "Rank", "Average"] + cols].to_csv(
        f"{dir_out_before}/{bench}.csv", index=False
    )

    sensitivity, weights = get_sensitivity(data, cols, return_weight=True)
    new_cols = ["%s (× %.4lf)" % (c, w) for c, w in zip(cols, weights)]

    weighted_df = data[cols].multiply(weights)
    weighted_df.columns = new_cols
    data = pd.concat([data, weighted_df], axis=1)

    data["Rank-Before"] = data["Rank"]
    data["Average-Before"] = data["Average"]
    data["Average"] = data[new_cols].mean(axis=1)
    data["Rank"] = rankdata(-data["Average"])
    data = data.sort_values("Rank")
    data = data.reset_index()
    data[["Model", "Rank", "Rank-Before", "Average", "Average-Before"] + new_cols].to_csv(
        f"{dir_out_after}/{bench}.csv", index=False
    )
    print(bench, "%.2lf, %.2lf" % (sensitivity[0], sensitivity[1]))

GLUE 0.12, 0.73
SuperGLUE 0.11, 0.30
OpenLLM 0.46, 0.89
MMLU 0.34, 0.93
BigBenchHard 0.25, 0.80
MTEB 0.18, 0.61
VTAB 0.07, 0.27


In [18]:
import os
import math
import numpy as np
from benchbench.data import load_ordinal_benchmark
from benchbench.utils.base import rankdata
from benchbench.utils.win_rate import WinningRate
from benchbench.measures.ordinal import get_sensitivity


ordinal_benchmark_list = [
    "BigCode",
    "HELM-accuracy",
    "HELM-fairness",
    "HELM-robustness",
    "HEIM-alignment_auto",
    "HEIM-quality_auto",
    "HEIM-aesthetics_auto",
    "HEIM-alignment_human",
    "HEIM-nudity",
    "HEIM-quality_human",
    "HEIM-aesthetics_human",
]

model_col_name = {
    "BigCode": "Models",
    "HELM-accuracy": "Model/adapter",
    "HELM-fairness": "Model",
    "HELM-robustness": "Model",
    "HEIM-alignment_auto": "Model/adapter",
    "HEIM-quality_auto": "Model/adapter",
    "HEIM-aesthetics_auto": "Model/adapter",
    "HEIM-alignment_human": "Model/adapter",
    "HEIM-nudity": "Model/adapter",
    "HEIM-quality_human": "Model/adapter",
    "HEIM-aesthetics_human": "Model/adapter",
}

dir_out_before = "./data/before"
dir_out_after = "./data/after"
os.makedirs(dir_out_before, exist_ok=True)
os.makedirs(dir_out_after, exist_ok=True)
for bench in ordinal_benchmark_list:
    data, cols = load_ordinal_benchmark(bench, do_rerank=True)
    cols = list(cols)
    inv_indices = np.arange(len(data) // 5)
    inv_data = data.iloc[inv_indices].copy()

    win_rate_calculator = WinningRate(data, cols)
    inv_data["WinRate"] = win_rate_calculator.get_winning_rate(inv_indices)
    inv_data["Rank"] = rankdata(-inv_data["WinRate"])
    mc = model_col_name[bench]
    inv_data["Model"] = inv_data[mc]
    inv_data["Model"] = inv_data["Model"].apply(lambda s: s.replace(",", " "))

    sensitivity, new_indices = get_sensitivity(
        data, cols, inv_indices=inv_indices, return_indices=True
    )
    new_data = data.iloc[new_indices].copy()

    new_data["Model"] = new_data[mc]
    new_data["Model"] = new_data["Model"].apply(lambda s: s.replace(",", " "))
    new_data["Rank-Before"] = [
        inv_data["Rank"].values[i] if i in inv_indices else math.nan for i in new_indices
    ]
    new_data["WinRate-Before"] = [
        inv_data["WinRate"].values[i] if i in inv_indices else math.nan for i in new_indices
    ]
    new_win_rate = win_rate_calculator.get_winning_rate(new_indices)[inv_indices]
    new_data["WinRate"] = [
        new_win_rate[i] if i in inv_indices else math.nan for i in new_indices
    ]
    new_rank = rankdata(-new_win_rate)
    new_data["Rank"] = [new_rank[i] if i in inv_indices else math.nan for i in new_indices]

    new_data = new_data.sort_values("Rank")
    new_data = new_data.reset_index()
    new_data[
        ["Model", "Rank", "Rank-Before", "WinRate", "WinRate-Before"] + cols
    ].to_csv(f"{dir_out_after}/{bench}.csv", index=False)

    inv_data = inv_data.sort_values("Rank")
    inv_data = inv_data.reset_index()
    inv_data[["Model", "Rank", "WinRate"] + cols].to_csv(
        f"{dir_out_before}/{bench}.csv", index=False
    )

    print(bench, "%.2lf, %.2lf" % (sensitivity[0], sensitivity[1]))


Episode 999, loss 1.92
BigCode 0.04, 0.14
Episode 999, loss 3.44
HELM-accuracy 0.32, 0.67
Episode 999, loss 4.67
HELM-fairness 0.18, 0.58
Episode 999, loss 3.70
HELM-robustness 0.26, 0.58
Episode 999, loss 0.60
HEIM-alignment_auto 0.30, 0.50
Episode 999, loss 1.00
HEIM-quality_auto 0.10, 0.25
Episode 999, loss 0.39
HEIM-aesthetics_auto 0.40, 0.50
Episode 999, loss 1.37
HEIM-alignment_human 0.10, 0.25
Episode 999, loss 0.26
HEIM-nudity 0.39, 0.62
Episode 999, loss 0.52
HEIM-quality_human 0.10, 0.25
Episode 999, loss 0.18
HEIM-aesthetics_human 0.10, 0.25


## V1: Adding new benchmarks

### Function definitions
One should run this first every time a new benchmark is added.

In [1]:
%pip install benchbench --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import pandas as pd
from benchbench.data import cardinal_benchmark_list
from benchbench.data import load_cardinal_benchmark
from benchbench.utils.base import rankdata
import benchbench.measures.cardinal as cardinal


def generate_cardinal(bench_name, data, cols, model_col):
    dir_out_before = "./data/before"
    dir_out_after = "./data/after"
    os.makedirs(dir_out_before, exist_ok=True)
    os.makedirs(dir_out_after, exist_ok=True)

    cols = list(cols)
    data["Average"] = data[cols].mean(axis=1)
    data["Rank"] = rankdata(-data["Average"])
    if model_col is None:
        data["Model"] = data.index
    else:
        data["Model"] = data[model_col]
    data["Model"] = data["Model"].apply(lambda s: s.replace(",", " "))
    data = data.sort_values("Rank")
    data = data.reset_index()
    data[["Model", "Rank", "Average"] + cols].to_csv(
        f"{dir_out_before}/{bench_name}.csv", index=False
    )

    sensitivity, weights = cardinal.get_sensitivity(data, cols, return_weight=True)
    new_cols = ["%s (× %.4lf)" % (c, w) for c, w in zip(cols, weights)]

    weighted_df = data[cols].multiply(weights)
    weighted_df.columns = new_cols
    data = pd.concat([data, weighted_df], axis=1)

    data["Rank-Before"] = data["Rank"]
    data["Average-Before"] = data["Average"]
    data["Average"] = data[new_cols].mean(axis=1)
    data["Rank"] = rankdata(-data["Average"])
    data = data.sort_values("Rank")
    data = data.reset_index()
    data[
        ["Model", "Rank", "Rank-Before", "Average", "Average-Before"] + new_cols
    ].to_csv(f"{dir_out_after}/{bench_name}.csv", index=False)

    diversity = cardinal.get_diversity(data, cols)
    ret1 = json.dumps(
        {
            "label": bench_name,
            "data": [
                {
                    "x": "%.2lf" % diversity[0],
                    "y": "%.2lf" % sensitivity[0],
                    "url": f"benchmark_detail.html?bench={bench_name}",
                }
            ],
            "pointRadius": 10,
        }
    )
    ret2 = json.dumps(
        {
            "label": bench_name,
            "data": [
                {
                    "x": "%.2lf" % diversity[0],
                    "y": "%.2lf" % sensitivity[1],
                    "url": f"benchmark_detail.html?bench={bench_name}",
                }
            ],
            "pointRadius": 10,
        }
    )
    print(ret1)
    print(ret2)

In [3]:
import os
import math
import numpy as np
from benchbench.data import load_ordinal_benchmark
from benchbench.utils.base import rankdata
from benchbench.utils.win_rate import WinningRate
import benchbench.measures.ordinal as ordinal


def generate_ordinal(bench_name, data, cols, model_col):
    dir_out_before = "./data/before"
    dir_out_after = "./data/after"
    os.makedirs(dir_out_before, exist_ok=True)
    os.makedirs(dir_out_after, exist_ok=True)

    cols = list(cols)
    inv_indices = np.arange(len(data) // 5)
    inv_data = data.iloc[inv_indices].copy()

    win_rate_calculator = WinningRate(data, cols)
    inv_data["WinRate"] = win_rate_calculator.get_winning_rate(inv_indices)
    inv_data["Rank"] = rankdata(-inv_data["WinRate"])
    inv_data["Model"] = inv_data[model_col]
    inv_data["Model"] = inv_data["Model"].apply(lambda s: s.replace(",", " "))

    sensitivity, new_indices = ordinal.get_sensitivity(
        data, cols, inv_indices=inv_indices, return_indices=True
    )
    new_data = data.iloc[new_indices].copy()

    new_data["Model"] = new_data[model_col]
    new_data["Model"] = new_data["Model"].apply(lambda s: s.replace(",", " "))
    new_data["Rank-Before"] = [
        inv_data["Rank"].values[i] if i in inv_indices else math.nan
        for i in new_indices
    ]
    new_data["WinRate-Before"] = [
        inv_data["WinRate"].values[i] if i in inv_indices else math.nan
        for i in new_indices
    ]
    new_win_rate = win_rate_calculator.get_winning_rate(new_indices)[inv_indices]
    new_data["WinRate"] = [
        new_win_rate[i] if i in inv_indices else math.nan for i in new_indices
    ]
    new_rank = rankdata(-new_win_rate)
    new_data["Rank"] = [
        new_rank[i] if i in inv_indices else math.nan for i in new_indices
    ]

    new_data = new_data.sort_values("Rank")
    new_data = new_data.reset_index()
    new_data[
        ["Model", "Rank", "Rank-Before", "WinRate", "WinRate-Before"] + cols
    ].to_csv(f"{dir_out_after}/{bench_name}.csv", index=False)

    inv_data = inv_data.sort_values("Rank")
    inv_data = inv_data.reset_index()
    inv_data[["Model", "Rank", "WinRate"] + cols].to_csv(
        f"{dir_out_before}/{bench_name}.csv", index=False
    )

    diversity = ordinal.get_diversity(data, cols)
    ret1 = json.dumps(
        {
            "label": bench_name,
            "data": [
                {
                    "x": "%.2lf" % diversity[0],
                    "y": "%.2lf" % sensitivity[0],
                    "url": f"benchmark_detail.html?bench={bench_name}",
                }
            ],
            "pointRadius": 10,
        }
    )
    ret2 = json.dumps(
        {
            "label": bench_name,
            "data": [
                {
                    "x": "%.2lf" % diversity[0],
                    "y": "%.2lf" % sensitivity[1],
                    "url": f"benchmark_detail.html?bench={bench_name}",
                }
            ],
            "pointRadius": 10,
        }
    )
    print(ret1)
    print(ret2)

### OpenLLM V2

In [4]:
import pandas as pd


def get_openllm_v2():
    raw = """T
    Model

    Average ⬆️
    IFEval
    BBH
    MATH Lvl 5
    GPQA
    MUSR
    MMLU-PRO
    💬
    Qwen/Qwen2-72B-Instruct 📑

    42.49
    79.89
    57.48
    35.12
    16.33
    17.17
    48.92
    💬
    meta-llama/Meta-Llama-3-70B-Instruct 📑

    36.18
    80.99
    50.19
    23.34
    4.92
    10.92
    46.74
    🟢
    Qwen/Qwen2-72B 📑

    35.13
    38.24
    51.86
    29.15
    19.24
    19.73
    52.56
    💬
    mistralai/Mixtral-8x22B-Instruct-v0.1 📑

    33.89
    71.84
    44.11
    18.73
    16.44
    13.49
    38.7
    💬
    HuggingFaceH4/zephyr-orpo-141b-A35b-v0.1 📑

    33.77
    65.11
    47.5
    18.35
    17.11
    14.72
    39.85
    💬
    microsoft/Phi-3-medium-4k-instruct 📑

    32.67
    64.23
    49.38
    16.99
    11.52
    13.05
    40.84
    💬
    01-ai/Yi-1.5-34B-Chat 📑

    32.63
    60.67
    44.26
    23.34
    15.32
    13.06
    39.12
    🔶
    alpindale/WizardLM-2-8x22B 📑

    32.61
    52.72
    48.58
    22.28
    17.56
    14.54
    39.96
    💬
    CohereForAI/c4ai-command-r-plus 📑

    30.86
    76.64
    39.92
    7.55
    7.38
    20.42
    33.24
    💬
    internlm/internlm2_5-7b-chat 📑

    30.46
    61.4
    57.67
    8.31
    10.63
    14.35
    30.42
    🟢
    Qwen/Qwen1.5-110B 📑

    29.56
    34.22
    44.28
    23.04
    13.65
    13.71
    48.45
    🔶
    abacusai/Smaug-72B-v0.1 📑

    29.56
    51.7
    42.42
    17.75
    9.62
    15.39
    40.46
    💬
    Qwen/Qwen1.5-110B-Chat 📑

    29.22
    59.39
    44.98
    0
    12.19
    16.29
    42.5
    💬
    microsoft/Phi-3-small-128k-instruct 📑

    28.75
    63.44
    45.57
    0
    9.84
    14.7
    38.94
    💬
    01-ai/Yi-1.5-9B-Chat 📑

    27.71
    60.46
    36.95
    11.63
    11.3
    12.84
    33.06
    💬
    Qwen/Qwen1.5-32B-Chat 📑

    27.1
    55.32
    44.55
    6.65
    7.49
    10.2
    38.41
    💬
    NousResearch/Nous-Hermes-2-Mixtral-8x7B-DPO 📑

    26.95
    58.7
    37.17
    10.27
    9.62
    16.28
    29.63
    💬
    deepseek-ai/deepseek-llm-67b-chat 📑

    26.87
    55.87
    33.23
    6.65
    8.84
    23.93
    32.71
    🔶
    MaziyarPanahi/Llama-3-8B-Instruct-v0.8 📑

    26.75
    75.12
    28.27
    7.1
    7.38
    10.92
    31.68
    🟢
    Qwen/Qwen1.5-32B 📑

    26.69
    32.97
    38.98
    26.66
    10.63
    12.04
    38.89
    🟢
    01-ai/Yi-1.5-34B-32K 📑

    26.4
    31.19
    43.38
    13.44
    15.1
    14.08
    41.21
    🟢
    meta-llama/Meta-Llama-3-70B 📑

    26.37
    16.03
    48.71
    16.54
    19.69
    16.01
    41.21
    💬
    microsoft/Phi-3-mini-4k-instruct 📑

    25.97
    56.13
    39.27
    11.63
    9.28
    7.64
    31.85
    🔶
    Sao10K/L3-8B-Stheno-v3.2 📑

    25.76
    68.73
    32.02
    8.53
    8.05
    6.45
    30.76
    🟢
    mistral-community/mixtral-8x22B-v0.3 📑

    25.55
    25.83
    45.73
    16.84
    17
    7.46
    40.44
    🟢
    mistralai/Mixtral-8x22B-v0.1 📑

    25.49
    25.83
    45.59
    16.84
    16.78
    7.46
    40.44
    💬
    microsoft/Phi-3-mini-128k-instruct 📑

    25.49
    59.76
    37.1
    8.91
    9.06
    7.71
    30.38
    🟢
    01-ai/Yi-1.5-34B 📑

    25.43
    28.41
    42.75
    14.05
    15.44
    11.22
    40.73
    💬
    CohereForAI/c4ai-command-r-v01 📑

    25.35
    67.48
    34.56
    0
    7.61
    16.13
    26.33
    💬
    databricks/dbrx-instruct 📑

    25.2
    54.16
    35.96
    6.87
    12.19
    12.2
    29.81
    🟢
    Qwen/Qwen2-57B-A14B 📑

    25.03
    31.13
    38.88
    18.66
    7.49
    10.54
    43.51
    💬
    Qwen/Qwen2-7B-Instruct 📑

    24.76
    56.79
    37.81
    8.61
    6.38
    7.37
    31.64
    🔶
    NousResearch/Hermes-2-Theta-Llama-3-8B 📑

    24.62
    65.18
    32.05
    8.69
    7.16
    8.36
    26.32
    💬
    CohereForAI/aya-23-35B 📑

    24.62
    64.62
    34.86
    2.64
    5.93
    13.47
    26.18
    🔶
    ZeusLabs/L3-Aethora-15B-V2 📑

    24.57
    72.08
    28.97
    7.33
    5.03
    6.25
    27.78
    💬
    mistralai/Mixtral-8x7B-Instruct-v0.1 📑

    24.35
    53.95
    34.02
    9.06
    7.61
    12.11
    29.36
    💬
    argilla/notux-8x7b-v1 📑

    24.13
    54.53
    32.86
    8.69
    7.49
    11.57
    29.66
    💬
    vicgalle/Roleplay-Llama-3-8B 📑

    23.94
    73.2
    28.55
    8.69
    1.45
    1.68
    30.09
    💬
    meta-llama/Meta-Llama-3-8B-Instruct 📑

    23.91
    74.08
    28.24
    8.69
    1.23
    1.6
    29.6
    💬
    01-ai/Yi-34B-Chat 📑

    23.9
    46.99
    37.62
    4.31
    11.74
    8.36
    34.37
    💬
    abacusai/Smaug-34B-v0.1 📑

    23.76
    50.16
    34.26
    0
    10.63
    8.13
    39.37
    🟢
    Qwen/Qwen2-7B 📑

    23.66
    31.49
    34.71
    18.81
    7.27
    14.32
    35.37
    💬
    NousResearch/Nous-Hermes-2-SOLAR-10.7B 📑

    23.32
    52.79
    34.99
    5.21
    5.82
    13.83
    27.31
    🔶
    UCLA-AGI/Llama-3-Instruct-8B-SPPO-Iter3 📑

    23.31
    68.28
    29.74
    7.33
    2.01
    3.09
    29.38
    💬
    google/gemma-2-9b-it 📑

    23.18
    76.95
    38.55
    0
    12.42
    9.38
    1.78
    💬
    UCLA-AGI/Llama-3-Instruct-8B-SPPO-Iter3 📑

    23.06
    67.03
    29.72
    7.18
    2.01
    2.89
    29.53
    💬
    MaziyarPanahi/Phi-3-mini-4k-instruct-v0.3 📑

    23.02
    49.26
    37.66
    2.95
    9.06
    7.75
    31.42
    💬
    LLM360/K2-Chat 📑

    22.93
    51.52
    33.79
    1.59
    7.49
    16.82
    26.34
    💬
    01-ai/Yi-1.5-9B-Chat-16K 📑

    22.9
    42.14
    31.5
    12.61
    7.83
    10.04
    33.26
    💬
    refuelai/Llama-3-Refueled 📑

    22.73
    46.2
    41.72
    3.93
    6.6
    14.64
    23.28
    🔶
    openchat/openchat-3.6-8b-20240522 📑

    22.69
    53.59
    33.23
    7.33
    9.06
    8.18
    24.77
    🔶
    openchat/openchat-3.5-0106 📑

    22.59
    59.51
    24.17
    6.5
    8.17
    11.75
    25.46
    💬
    openchat/openchat-3.5-1210 📑

    22.56
    60.37
    23.24
    6.87
    6.82
    14.28
    23.81
    🔶
    WizardLMTeam/WizardLM-70B-V1.0 📑

    22.32
    49.51
    37.54
    3.47
    2.13
    14.09
    27.18
    🟢
    01-ai/Yi-34B 📑

    22.26
    30.46
    35.54
    4.46
    15.55
    9.65
    37.91
    💬
    01-ai/Yi-1.5-6B-Chat 📑

    22.05
    48.02
    23.55
    12.54
    9.06
    14.7
    24.41
    🤝
    mlabonne/NeuralDaredevil-8B-abliterated 📑

    22.04
    45.6
    30.33
    8.01
    7.61
    9.08
    31.65
    🟢
    01-ai/Yi-1.5-9B 📑

    21.95
    29.36
    30.5
    10.2
    17.23
    12.03
    32.4
    💬
    NousResearch/Nous-Hermes-2-Mixtral-8x7B-SFT 📑

    21.78
    57.31
    30.59
    1.74
    6.94
    11.14
    22.96
    💬
    NousResearch/Hermes-2-Pro-Mistral-7B 📑

    21.64
    56.68
    29.43
    4.83
    3.13
    14.13
    21.63
    💬
    NousResearch/Hermes-2-Pro-Llama-3-8B 📑

    21.63
    53.62
    30.67
    5.74
    5.7
    11.25
    22.8
    💬
    openchat/openchat_3.5 📑

    21.52
    59.31
    21.58
    6.57
    6.49
    11.26
    23.93
    🔶
    Intel/neural-chat-7b-v3-2 📑

    21.43
    49.88
    30.24
    4.53
    5.37
    20.06
    18.52
    💬
    teknium/OpenHermes-2-Mistral-7B 📑

    21.33
    52.86
    29.25
    3.85
    4.47
    16.06
    21.46
    💬
    teknium/OpenHermes-2.5-Mistral-7B 📑

    21.22
    55.71
    27.77
    4.46
    4.47
    12.06
    22.83
    💬
    Qwen/Qwen1.5-14B-Chat 📑

    21.02
    47.68
    32.76
    0
    2.68
    13.93
    29.09
    💬
    NousResearch/Nous-Hermes-2-Mistral-7B-DPO 📑

    21.01
    57.63
    27.79
    4.23
    5.7
    8.33
    22.39
    🔶
    Intel/neural-chat-7b-v3-1 📑

    21
    46.87
    29.74
    3.17
    5.37
    22.24
    18.64
    💬
    berkeley-nest/Starling-LM-7B-alpha 📑

    20.64
    54.8
    21.95
    7.18
    6.26
    9.5
    24.13
    🟢
    Qwen/Qwen1.5-14B 📑

    20.22
    29.05
    30.06
    16.47
    5.93
    10.46
    29.37
    🔶
    Intel/neural-chat-7b-v3-3 📑

    19.99
    47.63
    27.75
    0.68
    5.26
    20.58
    18.05
    🟢
    01-ai/Yi-34B-200K 📑

    19.8
    15.42
    36.02
    4.46
    14.21
    9.41
    39.27
    💬
    upstage/SOLAR-10.7B-Instruct-v1.0 📑

    19.63
    47.37
    31.87
    0
    7.83
    6.94
    23.76
    🔶
    RLHFlow/LLaMA3-iterative-DPO-final 📑

    19.63
    53.34
    29.79
    0
    4.47
    5.08
    25.08
    🟢
    01-ai/Yi-1.5-9B-32K 📑

    19.61
    23.03
    28.94
    9.59
    14.54
    10.83
    30.72
    🟢
    mistralai/Mixtral-8x7B-v0.1 📑

    19.23
    23.5
    29.73
    8.84
    9.28
    12.55
    31.5
    💬
    mistralai/Mistral-7B-Instruct-v0.3 📑

    19.11
    54.65
    25.57
    3.17
    3.91
    4.3
    23.06
    💬
    HuggingFaceH4/zephyr-7b-alpha 📑

    18.52
    51.91
    23.89
    1.51
    6.38
    7.5
    19.94
    💬
    mistralai/Mistral-7B-Instruct-v0.2 📑

    18.44
    54.96
    22.91
    2.64
    3.47
    7.61
    19.08
    💬
    cognitivecomputations/dolphin-2.9-llama3-8b 📑

    18.3
    38.5
    27.86
    5.06
    4.92
    13.79
    19.68
    🟢
    meta-llama/Llama-2-70b-hf 📑

    18.25
    24.07
    35.9
    2.49
    7.05
    9.78
    30.2
    🔶
    microsoft/Orca-2-13b 📑

    18.14
    31.28
    27.31
    0.98
    4.03
    25.79
    19.44
    💬
    gradientai/Llama-3-8B-Instruct-Gradient-1048k 📑

    18.12
    44.56
    21.01
    4.38
    3.69
    13.52
    21.56
    🟢
    THUDM/glm-4-9b 📑

    18.01
    14.26
    35.81
    0
    8.84
    14.19
    34.94
    🔶
    Intel/neural-chat-7b-v3 📑

    17.94
    27.78
    30.21
    2.19
    5.59
    23.02
    18.87
    💬
    HuggingFaceH4/zephyr-7b-beta 📑

    17.72
    49.5
    21.49
    2.42
    5.37
    7.73
    19.79
    💬
    Open-Orca/Mistral-7B-OpenOrca 📑

    17.62
    49.78
    25.84
    2.95
    2.91
    5.89
    18.37
    🟢
    01-ai/Yi-9B 📑

    17.61
    27.09
    27.63
    4.38
    9.06
    8.91
    28.6
    🟢
    01-ai/Yi-9B-200K 📑

    17.59
    23.27
    26.49
    5.82
    8.72
    12.11
    29.13
    💬
    google/recurrentgemma-9b-it 📑

    17.59
    49.8
    19.99
    0.76
    3.47
    13.97
    17.53
    💬
    Deci/DeciLM-7B-instruct 📑

    17.43
    48.8
    23.89
    2.79
    5.26
    5.99
    17.87
    💬
    google/gemma-1.1-7b-it 📑

    17.4
    50.39
    15.93
    3.17
    5.82
    11.51
    17.6
    🟢
    upstage/SOLAR-10.7B-v1.0 📑

    16.77
    24.21
    29.79
    2.11
    4.14
    13.68
    26.67
    💬
    ibm/merlinite-7b 📑

    16.74
    24.99
    29.98
    2.34
    6.26
    13.88
    22.98
    💬
    Qwen/Qwen1.5-7B-Chat 📑

    16.58
    43.71
    22.38
    0
    7.05
    4.64
    21.68
    🔶
    CausalLM/14B 📑

    16.53
    27.88
    24.78
    3.32
    7.05
    11.47
    24.68
    🟢
    01-ai/Yi-1.5-6B 📑

    16.47
    26.11
    21.94
    5.66
    8.28
    13.51
    23.34
    💬
    stabilityai/stablelm-2-12b-chat 📑

    16.22
    40.82
    25.25
    2.04
    2.24
    7.73
    19.27
    💬
    CohereForAI/aya-23-8B 📑

    15.97
    46.99
    20.2
    1.44
    4.59
    8.42
    14.2
    💬
    HuggingFaceH4/zephyr-7b-gemma-v0.1 📑

    15.83
    33.64
    24.05
    6.65
    5.93
    4.18
    20.53"""

    data, cols = {}, []
    num_cols = 10

    for i, line in enumerate(raw.split("\n")):
        line = line.strip()
        if i < num_cols:
            cols.append(line)
            data[line] = []
        else:
            try:
                line = float(line)
            except (ValueError, SyntaxError, TypeError):
                pass
            data[cols[i % num_cols]].append(line)

    bench_name = "OpenLLM-v2"
    data = pd.DataFrame(data)
    cols = list(data.columns)[4:]
    model_col = "Model"

    avg = data[cols].values.mean(1)
    order = sorted(np.arange(len(data)), key=lambda x: -avg[x])
    data = data.iloc[order].reset_index(drop=True)

    return bench_name, data, cols, model_col

In [5]:
bench_name, data, cols, model_col = get_openllm_v2()
data

,T,Model,,Average ⬆️,IFEval,BBH,MATH Lvl 5,GPQA,MUSR,MMLU-PRO
0,💬,Qwen/Qwen2-72B-Instruct 📑,,42.49,79.89,57.48,35.12,16.33,17.17,48.92
1,💬,meta-llama/Meta-Llama-3-70B-Instruct 📑,,36.18,80.99,50.19,23.34,4.92,10.92,46.74
2,🟢,Qwen/Qwen2-72B 📑,,35.13,38.24,51.86,29.15,19.24,19.73,52.56
3,💬,mistralai/Mixtral-8x22B-Instruct-v0.1 📑,,33.89,71.84,44.11,18.73,16.44,13.49,38.70
4,💬,HuggingFaceH4/zephyr-orpo-141b-A35b-v0.1 📑,,33.77,65.11,47.50,18.35,17.11,14.72,39.85
...,...,...,...,...,...,...,...,...,...,...
95,🔶,CausalLM/14B 📑,,16.53,27.88,24.78,3.32,7.05,11.47,24.68
96,🟢,01-ai/Yi-1.5-6B 📑,,16.47,26.11,21.94,5.66,8.28,13.51,23.34
97,💬,stabilityai/stablelm-2-12b-chat 📑,,16.22,40.82,25.25,2.04,2.24,7.73,19.27
98,💬,CohereForAI/aya-23-8B 📑,,15.97,46.99,20.20,1.44,4.59,8.42,14.20


In [6]:
generate_cardinal(bench_name, data, cols, model_col)

{"label": "OpenLLM-v2", "data": [{"x": "0.59", "y": "0.41", "url": "benchmark_detail.html?bench=OpenLLM-v2"}], "pointRadius": 10}
{"label": "OpenLLM-v2", "data": [{"x": "0.59", "y": "0.83", "url": "benchmark_detail.html?bench=OpenLLM-v2"}], "pointRadius": 10}


### Helm Light

In [7]:
import pandas as pd
import numpy as np
from benchbench.utils.win_rate import WinningRate


def get_helm_light():
    raw = """Model
    Mean win rate
    NarrativeQA - F1
    NaturalQuestions (open) - F1
    NaturalQuestions (closed) - F1
    OpenbookQA - EM
    MMLU - EM
    MATH - Equivalent (CoT)
    GSM8K - EM
    LegalBench - EM
    MedQA - EM
    WMT 2014 - BLEU-4
    DBRX Instruct
    0.393
    0.488
    0.55
    0.284
    0.91
    0.643
    0.358
    0.671
    0.426
    0.694
    0.131
    DeepSeek LLM Chat (67B)
    0.62
    0.581
    0.733
    0.412
    0.88
    0.641
    0.615
    0.795
    0.637
    0.628
    0.186
    Falcon (40B)
    0.282
    0.671
    0.676
    0.392
    0.662
    0.507
    0.128
    0.267
    0.442
    0.419
    0.162
    Falcon (7B)
    0.089
    0.621
    0.58
    0.285
    0.26
    0.288
    0.044
    0.055
    0.346
    0.254
    0.094
    Gemma (7B)
    0.442
    0.752
    0.665
    0.336
    0.808
    0.571
    0.5
    0.559
    0.581
    0.513
    0.187
    Llama 2 (13B)
    0.311
    0.741
    0.64
    0.371
    0.634
    0.505
    0.102
    0.266
    0.591
    0.392
    0.167
    Llama 2 (70B)
    0.589
    0.763
    0.674
    0.46
    0.838
    0.58
    0.323
    0.567
    0.673
    0.618
    0.196
    Llama 2 (7B)
    0.209
    0.686
    0.612
    0.333
    0.544
    0.425
    0.097
    0.154
    0.502
    0.392
    0.144
    Llama 3 (70B)
    0.873
    0.798
    0.743
    0.475
    0.934
    0.695
    0.663
    0.805
    0.733
    0.777
    0.225
    Llama 3 (8B)
    0.495
    0.754
    0.681
    0.378
    0.766
    0.602
    0.391
    0.499
    0.637
    0.581
    0.183
    LLaMA (65B)
    0.435
    0.755
    0.672
    0.433
    0.754
    0.584
    0.257
    0.489
    0.48
    0.507
    0.189
    Mistral Instruct v0.3 (7B)
    0.276
    0.716
    0.68
    0.253
    0.79
    0.51
    0.289
    0.538
    0.331
    0.517
    0.142
    Mistral v0.1 (7B)
    0.387
    0.716
    0.687
    0.367
    0.776
    0.584
    0.297
    0.377
    0.58
    0.525
    0.16
    Mixtral (8x22B)
    0.822
    0.779
    0.726
    0.478
    0.882
    0.701
    0.656
    0.8
    0.708
    0.704
    0.209
    Mixtral (8x7B 32K seqlen)
    0.653
    0.767
    0.699
    0.427
    0.868
    0.649
    0.494
    0.622
    0.63
    0.652
    0.19
    OLMo (7B)
    0.076
    0.597
    0.603
    0.259
    0.222
    0.305
    0.029
    0.044
    0.341
    0.229
    0.097
    Phi-2
    0.238
    0.703
    0.68
    0.155
    0.798
    0.518
    0.255
    0.581
    0.334
    0.41
    0.038
    Qwen1.5 Chat (110B)
    0.676
    0.721
    0.739
    0.35
    0.922
    0.704
    0.568
    0.815
    0.624
    0.64
    0.192
    Qwen1.5 (14B)
    0.542
    0.711
    0.772
    0.3
    0.862
    0.626
    0.686
    0.693
    0.593
    0.515
    0.178
    Qwen1.5 (32B)
    0.664
    0.589
    0.777
    0.353
    0.932
    0.628
    0.733
    0.773
    0.636
    0.656
    0.193
    Qwen1.5 (72B)
    0.725
    0.601
    0.758
    0.417
    0.93
    0.647
    0.683
    0.799
    0.694
    0.67
    0.201
    Qwen1.5 (7B)
    0.358
    0.448
    0.749
    0.27
    0.806
    0.569
    0.561
    0.6
    0.523
    0.479
    0.153
    Qwen2 Instruct (72B)
    0.849
    0.727
    0.776
    0.39
    0.954
    0.769
    0.79
    0.92
    0.712
    0.746
    0.207
    Arctic Instruct
    0.453
    0.654
    0.586
    0.39
    0.828
    0.575
    0.519
    0.768
    0.588
    0.581
    0.172
    Yi (34B)
    0.691
    0.782
    0.775
    0.443
    0.92
    0.65
    0.375
    0.648
    0.618
    0.656
    0.172
    Yi (6B)
    0.322
    0.702
    0.748
    0.31
    0.8
    0.53
    0.126
    0.375
    0.519
    0.497
    0.117
    Jurassic-2 Jumbo (178B)
    0.291
    0.728
    0.65
    0.385
    0.688
    0.483
    0.103
    0.239
    0.533
    0.431
    0.114
    Jurassic-2 Grande (17B)
    0.236
    0.744
    0.627
    0.35
    0.614
    0.471
    0.064
    0.159
    0.468
    0.39
    0.102
    Luminous Base (13B)
    0.064
    0.633
    0.577
    0.197
    0.286
    0.243
    0.026
    0.028
    0.332
    0.26
    0.066
    Luminous Extended (30B)
    0.107
    0.684
    0.611
    0.253
    0.272
    0.248
    0.04
    0.075
    0.421
    0.276
    0.083
    Luminous Supreme (70B)
    0.198
    0.743
    0.656
    0.299
    0.284
    0.316
    0.078
    0.137
    0.452
    0.276
    0.102
    Claude v1.3
    0.665
    0.723
    0.699
    0.409
    0.908
    0.631
    0.54
    0.784
    0.629
    0.618
    0.219
    Claude Instant 1.2
    0.52
    0.616
    0.731
    0.343
    0.844
    0.631
    0.499
    0.721
    0.586
    0.559
    0.194
    Claude 2.0
    0.629
    0.718
    0.67
    0.428
    0.862
    0.639
    0.603
    0.583
    0.643
    0.652
    0.219
    Claude 2.1
    0.571
    0.677
    0.611
    0.375
    0.872
    0.643
    0.632
    0.604
    0.643
    0.644
    0.204
    Claude 3 Haiku (20240307)
    0.353
    0.244
    0.252
    0.144
    0.838
    0.662
    0.131
    0.699
    0.46
    0.702
    0.148
    Claude 3 Sonnet (20240229)
    0.451
    0.111
    0.072
    0.028
    0.918
    0.652
    0.084
    0.907
    0.49
    0.684
    0.218
    Claude 3 Opus (20240229)
    0.742
    0.351
    0.264
    0.441
    0.956
    0.768
    0.76
    0.924
    0.662
    0.775
    0.24
    Command
    0.398
    0.749
    0.777
    0.391
    0.774
    0.525
    0.236
    0.452
    0.578
    0.445
    0.088
    Command Light
    0.147
    0.629
    0.686
    0.195
    0.398
    0.386
    0.098
    0.149
    0.397
    0.312
    0.023
    Command R
    0.405
    0.742
    0.72
    0.352
    0.782
    0.567
    0.266
    0.551
    0.507
    0.555
    0.149
    Command R Plus
    0.567
    0.735
    0.711
    0.343
    0.828
    0.59
    0.403
    0.738
    0.672
    0.567
    0.203
    PaLM-2 (Bison)
    0.636
    0.718
    0.813
    0.39
    0.878
    0.608
    0.421
    0.61
    0.645
    0.547
    0.241
    PaLM-2 (Unicorn)
    0.749
    0.583
    0.674
    0.435
    0.938
    0.702
    0.674
    0.831
    0.677
    0.684
    0.26
    Mistral Small (2402)
    0.398
    0.519
    0.587
    0.304
    0.862
    0.593
    0.621
    0.734
    0.389
    0.616
    0.169
    Mistral Medium (2312)
    0.367
    0.449
    0.468
    0.29
    0.83
    0.618
    0.565
    0.706
    0.452
    0.61
    0.169
    Mistral Large (2402)
    0.429
    0.454
    0.485
    0.311
    0.894
    0.638
    0.75
    0.694
    0.479
    0.499
    0.182
    GPT-3.5 (text-davinci-003)
    0.551
    0.731
    0.77
    0.413
    0.828
    0.555
    0.449
    0.615
    0.622
    0.531
    0.191
    GPT-3.5 (text-davinci-002)
    0.442
    0.719
    0.71
    0.394
    0.796
    0.568
    0.428
    0.479
    0.58
    0.525
    0.174
    GPT-3.5 Turbo (0613)
    0.475
    0.655
    0.678
    0.335
    0.838
    0.614
    0.667
    0.501
    0.528
    0.622
    0.187
    GPT-4 Turbo (1106 preview)
    0.791
    0.727
    0.763
    0.435
    0.95
    0.699
    0.857
    0.668
    0.626
    0.817
    0.205
    GPT-4 (0613)
    0.935
    0.768
    0.79
    0.457
    0.96
    0.735
    0.802
    0.932
    0.713
    0.815
    0.211
    GPT-4 Turbo (2024-04-09)
    0.933
    0.761
    0.795
    0.482
    0.97
    0.711
    0.833
    0.824
    0.727
    0.783
    0.218
    GPT-4o (2024-05-13)
    0.973
    0.804
    0.803
    0.501
    0.966
    0.748
    0.829
    0.905
    0.733
    0.857
    0.231
    Palmyra X V2 (33B)
    0.716
    0.752
    0.752
    0.428
    0.878
    0.621
    0.58
    0.735
    0.644
    0.598
    0.239
    Palmyra X V3 (72B)
    0.791
    0.706
    0.685
    0.407
    0.938
    0.702
    0.723
    0.831
    0.709
    0.684
    0.262"""

    data, cols = {}, []
    num_cols = 12

    for i, line in enumerate(raw.split("\n")):
        line = line.strip()
        if i < num_cols:
            cols.append(line)
            data[line] = []
        else:
            try:
                line = float(line)
            except (ValueError, SyntaxError, TypeError):
                pass
            data[cols[i % num_cols]].append(line)

    bench_name = "HELM-Light"
    data = pd.DataFrame(data)
    cols = list(data.columns)[2:]
    model_col = "Model"

    wr = WinningRate(data, cols)
    win_rate = wr.get_winning_rate()
    order = sorted(np.arange(len(data)), key=lambda x: -win_rate[x])
    data = data.iloc[order].reset_index(drop=True)

    return bench_name, data, cols, model_col

In [8]:
bench_name, data, cols, model_col = get_helm_light()
data

,Model,Mean win rate,NarrativeQA - F1,NaturalQuestions (open) - F1,NaturalQuestions (closed) - F1,OpenbookQA - EM,MMLU - EM,MATH - Equivalent (CoT),GSM8K - EM,LegalBench - EM,MedQA - EM,WMT 2014 - BLEU-4
0,GPT-4o (2024-05-13),0.973,0.804,0.803,0.501,0.966,0.748,0.829,0.905,0.733,0.857,0.231
1,GPT-4 (0613),0.935,0.768,0.790,0.457,0.960,0.735,0.802,0.932,0.713,0.815,0.211
2,GPT-4 Turbo (2024-04-09),0.933,0.761,0.795,0.482,0.970,0.711,0.833,0.824,0.727,0.783,0.218
3,Llama 3 (70B),0.873,0.798,0.743,0.475,0.934,0.695,0.663,0.805,0.733,0.777,0.225
4,Qwen2 Instruct (72B),0.849,0.727,0.776,0.390,0.954,0.769,0.790,0.920,0.712,0.746,0.207
5,Mixtral (8x22B),0.822,0.779,0.726,0.478,0.882,0.701,0.656,0.800,0.708,0.704,0.209
6,GPT-4 Turbo (1106 preview),0.791,0.727,0.763,0.435,0.950,0.699,0.857,0.668,0.626,0.817,0.205
7,Palmyra X V3 (72B),0.791,0.706,0.685,0.407,0.938,0.702,0.723,0.831,0.709,0.684,0.262
8,PaLM-2 (Unicorn),0.749,0.583,0.674,0.435,0.938,0.702,0.674,0.831,0.677,0.684,0.260
9,Claude 3 Opus (20240229),0.742,0.351,0.264,0.441,0.956,0.768,0.760,0.924,0.662,0.775,0.240


In [9]:
generate_ordinal(bench_name, data, cols, model_col)

Episode 999, loss 4.83
{"label": "HELM-Light", "data": [{"x": "0.37", "y": "0.17", "url": "benchmark_detail.html?bench=HELM-Light"}], "pointRadius": 10}
{"label": "HELM-Light", "data": [{"x": "0.37", "y": "0.50", "url": "benchmark_detail.html?bench=HELM-Light"}], "pointRadius": 10}
